# Preparação dos Dados - FlowCarreiras

Este notebook documenta e valida a preparação das duas bases abertas utilizadas no projeto:

- **Mapa Cultural de Pernambuco:** recorte de 1.000 agentes culturais extraídos da API pública.
- **contempArt:** 441 artistas em início de carreira ligados a escolas de arte alemãs.

As bases são complementares, mas possuem contextos e unidades de análise diferentes. Portanto, são preparadas e analisadas separadamente.

## 1. Configuração e caminhos

Os scripts reproduzíveis de extração, limpeza e criação de variáveis estão em `analise-dados/src/`. Este notebook utiliza os arquivos gerados por esses scripts para demonstrar e validar o processo.

In [ ]:
from pathlib import Path
import json

import numpy as np
import pandas as pd

try:
    from IPython.display import display
except ImportError:
    display = print

pd.set_option("display.max_columns", 50)
pd.set_option("display.max_colwidth", 100)

cwd = Path.cwd()
if (cwd / "analise-dados").exists():
    DATA_DIR = cwd / "analise-dados" / "data"
elif cwd.name == "notebooks":
    DATA_DIR = cwd.parent / "data"
else:
    DATA_DIR = cwd / "data"

RAW_DIR = DATA_DIR / "raw"
PROCESSED_DIR = DATA_DIR / "processed"
print(f"Diretório de dados: {DATA_DIR.resolve()}")

## 2. Metadados da extração do Mapa Cultural PE

O JSON bruto preserva a fonte, os parâmetros, o horário da extração e as páginas consultadas. Isso permite auditar e reproduzir o recorte utilizado.

In [ ]:
mapa_raw_path = RAW_DIR / "mapa_cultural_pe_agentes.json"
mapa_raw = json.loads(mapa_raw_path.read_text(encoding="utf-8"))

metadados_extracao = pd.DataFrame({
    "informacao": ["fonte", "extraido_em_utc", "total_registros", "paginas_consultadas"],
    "valor": [
        mapa_raw["fonte"],
        mapa_raw["extraido_em_utc"],
        mapa_raw["total_registros"],
        len(mapa_raw["urls_consultadas"]),
    ],
})
display(metadados_extracao)

## 3. Carregamento das bases limpas e enriquecidas

A versão **limpa** preserva os dados tratados próximos às fontes. A versão **enriquecida** adiciona variáveis derivadas criadas pelo grupo, sem substituir os arquivos limpos.

In [ ]:
mapa_limpo = pd.read_csv(PROCESSED_DIR / "mapa_cultural_pe_agentes.csv")
mapa_enriquecido = pd.read_csv(PROCESSED_DIR / "mapa_cultural_pe_agentes_enriquecido.csv")
contempart_limpo = pd.read_csv(PROCESSED_DIR / "contempart_artists.csv")
contempart_enriquecido = pd.read_csv(PROCESSED_DIR / "contempart_artists_enriquecido.csv")

resumo_bases = pd.DataFrame([
    ["Mapa Cultural PE - limpo", *mapa_limpo.shape],
    ["Mapa Cultural PE - enriquecido", *mapa_enriquecido.shape],
    ["contempArt - limpo", *contempart_limpo.shape],
    ["contempArt - enriquecido", *contempart_enriquecido.shape],
], columns=["base", "linhas", "colunas"])
display(resumo_bases)

## 4. Validação de identificadores e duplicidades

Cada base possui um identificador próprio. Duplicidades são verificadas separadamente por `id` e `artist_id`.

In [ ]:
validacao_ids = pd.DataFrame([
    ["Mapa Cultural PE", len(mapa_limpo), mapa_limpo["id"].nunique(), mapa_limpo["id"].duplicated().sum()],
    ["contempArt", len(contempart_limpo), contempart_limpo["artist_id"].nunique(), contempart_limpo["artist_id"].duplicated().sum()],
], columns=["base", "registros", "identificadores_unicos", "duplicidades"])
display(validacao_ids)

## 5. Valores ausentes

Valores ausentes são mantidos como ausentes. Eles não são substituídos por zero, média ou categorias inventadas, pois isso poderia distorcer as análises.

In [ ]:
def resumo_ausentes(dataframe):
    total = dataframe.isna().sum()
    percentual = (total / len(dataframe) * 100).round(2)
    return pd.DataFrame({"ausentes": total, "percentual": percentual}).sort_values("percentual", ascending=False)

print("Mapa Cultural PE")
display(resumo_ausentes(mapa_limpo))

print("contempArt")
display(resumo_ausentes(contempart_limpo))

## 6. Conversão e validação das datas

As datas são convertidas para `datetime`. Valores inválidos se tornam ausentes e atualizações anteriores à criação são contabilizadas como inconsistências.

In [ ]:
datas = mapa_limpo[["data_criacao", "data_atualizacao"]].copy()
datas["data_criacao"] = pd.to_datetime(datas["data_criacao"], errors="coerce")
datas["data_atualizacao"] = pd.to_datetime(datas["data_atualizacao"], errors="coerce")

validacao_datas = pd.DataFrame({
    "metrica": ["datas de criação inválidas/ausentes", "datas de atualização inválidas/ausentes", "atualizações anteriores à criação"],
    "quantidade": [
        datas["data_criacao"].isna().sum(),
        datas["data_atualizacao"].isna().sum(),
        (datas["data_atualizacao"] < datas["data_criacao"]).sum(),
    ],
})
display(validacao_datas)

## 7. Listas de áreas e tags

As listas do Mapa Cultural PE foram padronizadas com o separador ` | `. A separação em múltiplas linhas deve ocorrer apenas durante análises de frequência.

In [ ]:
amostra_termos = mapa_limpo.loc[
    mapa_limpo["termos_areas"].notna(),
    ["nome", "termos_areas", "termos_tags", "termos_subareas"],
].head(10)
display(amostra_termos)

## 8. Variáveis derivadas

As variáveis derivadas transformam dados existentes em indicadores úteis para EDA, regressão, classificação e dashboard.

In [ ]:
derivadas_mapa = ["possui_descricao", "quantidade_areas", "ano_criacao", "atualizacao_posterior"]
derivadas_contempart = ["possui_instagram", "possui_website", "taxa_engajamento", "nivel_visibilidade"]

print("Mapa Cultural PE - variáveis derivadas")
display(mapa_enriquecido[["id", "nome", *derivadas_mapa]].head(10))

print("contempArt - variáveis derivadas")
display(contempart_enriquecido[["artist_id", "full_name", *derivadas_contempart]].head(10))

In [ ]:
resumo_derivadas = pd.DataFrame({
    "metrica": [
        "Mapa PE: percentual com descrição",
        "Mapa PE: mediana de áreas por agente",
        "contempArt: percentual com Instagram",
        "contempArt: percentual com website",
        "contempArt: mediana da taxa de engajamento calculável",
    ],
    "valor": [
        f"{mapa_enriquecido['possui_descricao'].mean() * 100:.2f}%",
        mapa_enriquecido["quantidade_areas"].median(),
        f"{contempart_enriquecido['possui_instagram'].mean() * 100:.2f}%",
        f"{contempart_enriquecido['possui_website'].mean() * 100:.2f}%",
        contempart_enriquecido["taxa_engajamento"].median(),
    ],
})
display(resumo_derivadas)

## 9. Checklist final da preparação

- Bases brutas preservadas para auditoria.
- Bases limpas mantidas separadamente.
- Identificadores únicos e duplicidades verificados.
- Ausentes preservados e documentados.
- Datas convertidas e inconsistências temporais verificadas.
- Listas de áreas e tags padronizadas.
- Variáveis derivadas geradas em arquivos enriquecidos separados.
- Bases prontas para os notebooks de EDA, modelagem e dashboard.

### Limitação principal

O Mapa Cultural PE utiliza um recorte dos primeiros 1.000 registros retornados pela API. O recorte é adequado ao escopo acadêmico, mas não deve ser tratado como amostra aleatória representativa dos mais de 61 mil agentes cadastrados.